# Úvod do strojového učení

Cvičení navazuje na přednášku o základech strojového učení a má stejné čtyři části:

1. **Data** – vzorky, příznaky, cílová proměnná, data s označením a bez něj; základy práce s tabulkou v pandas.
2. **Principy** – model jako funkce naučená z dat, `fit` / `predict`, trénovací, validační a testovací data, podučení a přeučení.
3. **Úlohy** – regrese, klasifikace a shlukování na reálných datech.
4. **Evaluace** – baseline, metriky pro jednotlivé úlohy a proč samotná správnost (accuracy) může klamat.

Algoritmy z knihovny **scikit-learn** bereme jako hotové černé skříňky; jak fungují uvnitř, probereme na dalších cvičeních.
Stejně tak **předzpracování dat** (cvičení 08) a **systematický výběr modelu** – ladění hyperparametrů podle metriky, křížová validace, volba rozhodovacího prahu (cvičení 07).
Dnes proto používáme jen vestavěné datasety, které už jsou čisté.

Kód je okomentovaný řádek po řádku. Buňky označené **✏️ Úkol** řešíte během cvičení,
ve druhé půlce cvičení následuje [samostatná práce](Independent_Work_CZ.ipynb).

In [ ]:
import numpy as np                     # pole a matematické operace
import pandas as pd                    # tabulky DataFrame a sloupce Series
import matplotlib.pyplot as plt        # grafy a obrázky
from IPython.display import display    # hezké zobrazení tabulek uprostřed buňky

from sklearn import datasets, metrics                                     # vestavěná data a metriky
from sklearn.model_selection import train_test_split                      # dělení dat
from sklearn.dummy import DummyRegressor, DummyClassifier                 # triviální srovnávací modely
from sklearn.linear_model import LinearRegression                         # lineární regrese
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier    # rozhodovací stromy
from sklearn.ensemble import RandomForestClassifier                       # náhodný les
from sklearn.cluster import KMeans                                        # shlukování

RSTATE = 42  # pevné semínko náhodného generátoru -> opakovatelné výsledky

# Část 1 – Data

Strojové učení se učí z **dat**. Typický (tabulkový) dataset je tabulka, ve které

* každý **řádek** je jeden **vzorek** (pozorování, příklad) – např. jeden pacient,
* každý **sloupec** je jeden **příznak** (atribut, vstupní proměnná, anglicky *feature*) – např. velikost nádoru,
* volitelně je jeden zvláštní sloupec **cílová proměnná** (označení, anglicky *target* nebo *label*) – hodnota, kterou chceme předpovídat.

Vstupy zapisujeme jako matici $X$ tvaru `(počet_vzorků, počet_příznaků)` a cílové hodnoty jako vektor $y$ délky `počet_vzorků`.

## 1.1 Tabulkový dataset – Breast Cancer Wisconsin

Každý vzorek popisuje buněčná jádra změřená na mikroskopickém snímku vzorku prsní tkáně (569 vzorků, 30 číselných příznaků).
Cílová proměnná říká, zda je nádor *zhoubný* (malignant) nebo *nezhoubný* (benign).

S parametrem `as_frame=True` dostaneme data jako **pandas** objekty:

* `cancer.data` – jen příznaky (**DataFrame** = tabulka se jmény sloupců a indexem řádků),
* `cancer.target` – jen cílová proměnná (**Series** = jeden sloupec s indexem).

V datasetu je `0 = zhoubný` a `1 = nezhoubný`. Je zvykem kódovat hledanou třídu (nemoc) jako `1` – tzv. **pozitivní třídu** – proto kódování otočíme.

In [ ]:
cancer = datasets.load_breast_cancer(as_frame=True)    # načte data jako pandas objekty
X_cancer = cancer.data                                  # DataFrame s 30 příznaky
y_cancer = (cancer.target == 0).astype(int)             # Series: 1 = zhoubný (pozitivní třída), 0 = nezhoubný

df = X_cancer.assign(target=y_cancer)                   # nová tabulka: příznaky + cílový sloupec (jen pro prohlížení)
print("Tvar X:", X_cancer.shape, "  tvar y:", y_cancer.shape, "  tvar df:", df.shape)
display(df.head(3))                                     # první tři řádky

## 1.2 pandas v kostce

Pro prohlížení dat stačí pár příkazů:

| příkaz | co vrací |
|---|---|
| `df.shape`, `df.head()`, `df.dtypes` | rozměry, první řádky, datové typy sloupců |
| `df["a"]` / `df[["a", "b"]]` | jeden sloupec jako Series / výběr sloupců jako DataFrame |
| `df.loc[maska, ["a", "b"]]` | řádky, kde je maska `True`, a vybrané sloupce |
| `(podm1) & (podm2)`, `|`, `~` | „a zároveň“, „nebo“, „ne“ pro masky (každou podmínku dejte do závorek) |
| `y.value_counts(normalize=True)` | četnosti (podíly) jednotlivých hodnot |
| `df.describe()` | počet, průměr, směrodatná odchylka, minimum, kvartily, maximum |
| `df.drop(columns="target")` | tabulka bez daného sloupce |

Pozor: cílový sloupec do vstupů modelu **nepatří** – model by „opisoval“ správnou odpověď.

In [ ]:
radius = df["mean radius"]                     # jeden sloupec -> Series, tvar (n,)
radius_tab = df[["mean radius"]]               # seznam sloupců -> DataFrame, tvar (n, 1)
print(type(radius).__name__, radius.shape, "|", type(radius_tab).__name__, radius_tab.shape)

maska = (df["target"] == 1) & (df["mean radius"] > 20)          # zhoubné a zároveň velké nádory
print("Vybraných řádků:", maska.sum())                           # True se při součtu počítá jako 1
display(df.loc[maska, ["mean radius", "mean area", "target"]].head())   # řádky podle masky, vybrané sloupce

In [ ]:
cetnosti = pd.DataFrame({                                        # tabulka ze dvou Series se stejným indexem
    "počet": y_cancer.value_counts(),                            # kolik vzorků má každá třída
    "podíl": y_cancer.value_counts(normalize=True).round(3),     # totéž jako podíl
}).rename(index={0: "nezhoubný (0)", 1: "zhoubný (1)"})          # čitelné názvy místo kódů
display(cetnosti)

statistiky = X_cancer.describe().T                               # .T otočí tabulku: příznaky v řádcích
display(statistiky.loc[["mean radius", "mean area", "mean smoothness"], ["mean", "std", "min", "max"]])

**✏️ Úkol 1.1** – odpovězte podle výstupů výše:

* Kolik má dataset vzorků a kolik příznaků? Jsou všechny příznaky číselné?
* Je dataset *vyvážený* (jsou obě třídy zastoupeny stejně)?
* Porovnejte rozsahy `mean area` a `mean smoothness`. (Některým algoritmům rozdílné rozsahy vadí – řešit to budeme na cvičení 08; modelům, které použijeme dnes, nevadí.)
* Upravte masku tak, aby vybrala **nezhoubné** nádory s `mean radius` větším než 15. Kolik jich je?

## 1.3 Nejen tabulky – i obrázky jsou data

Šedotónový obrázek o $8 \times 8$ pixelech je jen $64$ čísel. Když ho *rozbalíme* do řádku, každý pixel se stane jedním příznakem a celý obrázek jedním řádkem matice $X$.

In [ ]:
digits = datasets.load_digits()                                   # 1797 ručně psaných číslic 0–9
print("Tvar X:", digits.data.shape, "  (1797 obrázků, každý 64 pixelů)")

fig, axes = plt.subplots(1, 10, figsize=(12, 2))                  # deset obrázků vedle sebe
for ax, img, label in zip(axes, digits.images, digits.target):    # obrázek a jeho správná třída
    ax.imshow(img, cmap="gray_r")                                 # 0 = bílá, 16 = černá
    ax.set_title(f"y = {label}")
    ax.axis("off")                                                # bez os
plt.show()

print(digits.images[0].astype(int))   # první obrázek jako matice 8 × 8 ...
print(digits.data[0].astype(int))     # ... a jako jeden řádek matice X (64 příznaků)

## 1.4 Data s označením a bez označení

* **Učení s učitelem** (supervised learning) – každý vzorek má známou cílovou hodnotu $y$. Model se učí zobrazení $X \rightarrow y$ (regrese, klasifikace).
* **Učení bez učitele** (unsupervised learning) – máme jen $X$. Model hledá strukturu přímo v datech (shlukování, snižování dimenze).

Oba grafy ukazují stejné dva příznaky. Vlevo označení známe, vpravo ne – to je vše, co vidí algoritmus učení bez učitele.

In [ ]:
f1, f2 = "mean radius", "mean texture"                            # dva příznaky pro 2D graf

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for cls, name, color in [(0, "nezhoubný", "tab:blue"), (1, "zhoubný", "tab:red")]:
    m = y_cancer == cls                                           # maska jedné třídy
    axes[0].scatter(X_cancer.loc[m, f1], X_cancer.loc[m, f2], s=12, alpha=0.6, c=color, label=name)
axes[0].set_title("Data s označením (učení s učitelem)")
axes[0].legend()

axes[1].scatter(X_cancer[f1], X_cancer[f2], s=12, alpha=0.6, c="gray")   # bez cílové proměnné
axes[1].set_title("Data bez označení (učení bez učitele)")

for ax in axes:
    ax.set_xlabel(f1)
axes[0].set_ylabel(f2)
plt.tight_layout()
plt.show()

# Část 2 – Principy

## 2.1 Model je funkce s naučitelnými parametry

Model je funkce $\hat{y} = f(x; \theta)$. **Učení** (trénování) znamená najít parametry $\theta$ tak, aby předpovědi $\hat{y}$ byly na trénovacích datech blízko skutečným hodnotám $y$
– tedy minimalizovat **ztrátovou funkci** (loss function).

* **Parametry** se model učí z dat (např. váhy lineárního modelu).
* **Hyperparametry** volíme my před trénováním (např. maximální hloubku rozhodovacího stromu).

Ve scikit-learn má každý model stejné rozhraní:

```python
model = NejakyModel(hyperparametry)  # 1. zvolíme model
model.fit(X_train, y_train)          # 2. naučíme parametry z dat
y_pred = model.predict(X_new)        # 3. použijeme model na nová data
```

Minimální příklad: vygenerujeme data podle $y = 3x + 2 + \text{šum}$ a necháme lineární regresi najít parametry.

In [ ]:
rng = np.random.default_rng(RSTATE)               # opakovatelný generátor náhodných čísel
x = rng.uniform(0, 10, size=50)                   # 50 vstupů z intervalu 0–10
y = 3 * x + 2 + rng.normal(0, 2, size=50)         # skutečná závislost + šum

model = LinearRegression()                        # 1. zvolíme model
model.fit(x.reshape(-1, 1), y)                    # 2. učení; sklearn chce X jako 2D matici (vzorky, příznaky)
print(f"Naučené parametry: směrnice = {model.coef_[0]:.2f}, posun = {model.intercept_:.2f}")
print("Předpověď pro x = 4:", model.predict([[4]]))   # 3. předpověď pro nový vzorek

xs = np.linspace(0, 10, 100).reshape(-1, 1)       # hustá mřížka pro vykreslení přímky
plt.scatter(x, y, s=15, label="data")
plt.plot(xs, model.predict(xs), "r", label="naučený model")
plt.xlabel("x"); plt.ylabel("y"); plt.legend()
plt.show()

## 2.2 Generalizace a rozdělení na trénovací a testovací data

Nechceme model, který je dobrý na datech, která už viděl – chceme model, který funguje na **nových, dosud neviděných datech**. Této schopnosti říkáme **generalizace**.

Abychom ji mohli odhadnout, schováme před trénováním část dat:

* **trénovací množina** – na ní se model učí,
* **testovací množina** – na ní model jen vyhodnotíme; při učení ji nesmí vidět.

`X` a `y` dělíme **jedním** voláním `train_test_split`, takže řádky vstupů a odpovědí zůstanou spárované. `stratify=y` zajistí, že poměr tříd bude v obou částech stejný.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(              # 80 % trénovací, 20 % testovací data
    X_cancer, y_cancer, test_size=0.2, stratify=y_cancer, random_state=RSTATE
)
print("Trénovací:", X_train.shape, " Testovací:", X_test.shape)
print(f"Podíl zhoubných – trénovací: {y_train.mean():.3f}, testovací: {y_test.mean():.3f}")  # průměr 0/1 = podíl jedniček

## 2.3 Podučení a přeučení

* **Podučení** (underfitting) – model je příliš jednoduchý a nedokáže zachytit závislost v datech. Je špatný na trénovacích *i* testovacích datech.
* **Přeučení** (overfitting) – model je příliš složitý a naučí se trénovací data nazpaměť včetně šumu. Na trénovacích datech je výborný, na testovacích špatný.

Příklad: 20 zašuměných trénovacích bodů (a 200 testovacích bodů ze stejného rozsahu) vygenerovaných ze sinusovky, proložených polynomy rostoucího stupně (= rostoucí složitost modelu).

In [ ]:
rng = np.random.default_rng(1)                                     # samostatný generátor pro tento příklad
def true_fn(x):                                                    # skutečná závislost bez šumu
    return np.sin(2 * np.pi * x)

x_tr = np.linspace(0, 1, 20);  y_tr = true_fn(x_tr) + rng.normal(0, 0.25, 20)     # malá trénovací data
x_te = rng.uniform(0, 1, 200); y_te = true_fn(x_te) + rng.normal(0, 0.25, 200)    # nezávislá testovací data
xs = np.linspace(0, 1, 500)                                        # mřížka pro hladké křivky

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, degree, title in zip(axes, [1, 4, 15], ["podučení", "dobrý model", "přeučení"]):
    poly = np.polynomial.Polynomial.fit(x_tr, y_tr, degree)        # proložení polynomem daného stupně
    mse_tr = np.mean((poly(x_tr) - y_tr) ** 2)                     # střední kvadratická chyba na trénovacích datech
    mse_te = np.mean((poly(x_te) - y_te) ** 2)                     # ... a na testovacích datech
    ax.scatter(x_tr, y_tr, s=20, label="trénovací data")
    ax.plot(xs, true_fn(xs), "g--", label="skutečná funkce")
    ax.plot(xs, poly(xs), "r", label=f"polynom stupně {degree}")
    ax.set_ylim(-2, 2)                                             # křivky mimo rozsah se oříznou
    ax.set_title(f"{title}\ntrénovací MSE = {mse_tr:.3f}, testovací MSE = {mse_te:.3f}")
    ax.legend(loc="lower left", fontsize=8)
plt.show()

## 2.4 Volba složitosti modelu – proč potřebujeme validační množinu

Jak zvolit vhodnou složitost (hyperparametr)? **Ne podle testovací množiny** – pak by už nebyla „neviděná“ a náš odhad by byl příliš optimistický.
Proto trénovací data rozdělíme ještě jednou:

| část | k čemu slouží |
|---|---|
| **trénovací** | učení parametrů modelu |
| **validační** | porovnání nastavení (hyperparametrů) a modelů |
| **testovací** | konečný odhad výkonu na neviděných datech |

Ukázka: rozhodovací stromy s různou maximální hloubkou na datasetu číslic. Graf jen **čteme očima** – systematický výběr modelu podle metriky a křížovou validaci probereme na cvičení 07.

In [ ]:
Xg_train, Xg_test, yg_train, yg_test = train_test_split(           # 1. odložíme testovací množinu
    digits.data, digits.target, test_size=0.2, stratify=digits.target, random_state=RSTATE
)
X_tr, X_val, y_tr, y_val = train_test_split(                       # 2. zbytek rozdělíme na trénovací a validační
    Xg_train, yg_train, test_size=0.25, stratify=yg_train, random_state=RSTATE
)

depths = range(1, 21)                                              # zkoušené maximální hloubky
acc_tr, acc_val = [], []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=RSTATE).fit(X_tr, y_tr)   # učení jen na trénovacích datech
    acc_tr.append(tree.score(X_tr, y_tr))                          # .score() = správnost (accuracy)
    acc_val.append(tree.score(X_val, y_val))                       # správnost na validačních datech

plt.plot(depths, acc_tr, "o-", label="trénovací data")
plt.plot(depths, acc_val, "o-", label="validační data")
plt.xticks(list(depths))
plt.xlabel("max_depth (složitost modelu)"); plt.ylabel("správnost (accuracy)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

**✏️ Úkol 2.1**

* Kde je model podučený a kde přeučený? Od jaké hloubky už se validační správnost nezlepšuje?
* Proč u hlubokých stromů dosáhne trénovací správnost hodnoty 1,0? Roste s ní i validační správnost?
* Změňte `random_state` při dělení na trénovací a validační data. Změní se tvar křivek? Co to říká o jediném validačním rozdělení?

# Část 3 – Úlohy

| úloha | typ učení | cílová proměnná | příklad |
|---|---|---|---|
| **Regrese** | s učitelem | spojité číslo | cena domu, vývoj nemoci |
| **Klasifikace** | s učitelem | diskrétní třída | zhoubný / nezhoubný, číslice 0–9 |
| **Shlukování** | bez učitele | žádná | skupiny podobných zákazníků |

Další úlohy, se kterými se můžete setkat: snižování dimenze, detekce anomálií, doporučování, generativní modelování, posilované učení.

## 3.1 Regrese – dataset Diabetes

10 vstupních měření (věk, pohlaví, BMI, krevní tlak a šest hodnot z krevního séra) u 442 pacientů. Cílová proměnná je číselná míra postupu nemoci po jednom roce.
Příznaky už autoři datasetu standardizovali.

In [ ]:
diabetes = datasets.load_diabetes(as_frame=True)                  # regresní data jako pandas
X_diab, y_diab = diabetes.data, diabetes.target                   # vstupy a spojitá cílová proměnná
display(X_diab.assign(target=y_diab).head(3))                     # prohlédneme si pár řádků

Xd_train, Xd_test, yd_train, yd_test = train_test_split(          # u spojitého cíle se nestratifikuje
    X_diab, y_diab, test_size=0.2, random_state=RSTATE
)
lin_reg = LinearRegression().fit(Xd_train, yd_train)                                  # lineární model
tree_reg = DecisionTreeRegressor(max_depth=3, random_state=RSTATE).fit(Xd_train, yd_train)  # strom hloubky 3

pd.DataFrame({                                                    # skutečnost a předpovědi vedle sebe
    "skutečná hodnota": yd_test.values[:5],
    "lineární regrese": lin_reg.predict(Xd_test[:5]).round(1),
    "rozhodovací strom": tree_reg.predict(Xd_test[:5]).round(1),
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, model, name in zip(axes, [lin_reg, tree_reg], ["Lineární regrese", "Rozhodovací strom (hloubka 3)"]):
    ax.scatter(yd_test, model.predict(Xd_test), s=15, alpha=0.7)  # skutečnost proti předpovědi
    lims = [y_diab.min(), y_diab.max()]
    ax.plot(lims, lims, "r--", label="dokonalá předpověď")        # body na úhlopříčce = bez chyby
    ax.set_title(name); ax.set_xlabel("skutečná hodnota"); ax.legend()
axes[0].set_ylabel("předpovězená hodnota")
plt.show()

Čím blíž jsou body červené úhlopříčce, tím lepší jsou předpovědi. *O kolik* je jeden model lepší než druhý, změříme v části 4.

Rozhodovací strom s hloubkou 3 má nejvýše $2^3 = 8$ listů, a umí tedy předpovědět nejvýše 8 různých hodnot – proto jsou vidět vodorovné „pruhy“.

## 3.2 Klasifikace

### Binární klasifikace – Breast Cancer

Klasifikátory nevracejí jen třídu (`predict`), ale často i **odhad pravděpodobnosti** jednotlivých tříd (`predict_proba`).
Sloupce `predict_proba` jsou ve stejném pořadí jako `model.classes_`.

In [ ]:
forest = RandomForestClassifier(n_estimators=200, random_state=RSTATE).fit(X_train, y_train)  # les 200 stromů
print("Pořadí tříd:", forest.classes_)                            # [0 1] -> sloupec 1 patří třídě 1 (zhoubný)

pd.DataFrame({
    "skutečná třída": y_test.values[:8],                          # správné odpovědi
    "předpovězená třída": forest.predict(X_test[:8]),             # předpovězené třídy
    "P(zhoubný)": forest.predict_proba(X_test[:8])[:, 1].round(3),  # odhad pravděpodobnosti třídy 1
})

### Klasifikace do více tříd – ručně psané číslice

Stejné rozhraní funguje i pro 10 tříd. Použijeme rozdělení číslic z části 2 a podíváme se na obrázky, ve kterých se model spletl.

In [ ]:
digit_clf = RandomForestClassifier(n_estimators=200, random_state=RSTATE).fit(Xg_train, yg_train)
yg_pred = digit_clf.predict(Xg_test)                              # předpovědi pro testovací obrázky

wrong = np.where(yg_pred != yg_test)[0]                           # pozice chybně klasifikovaných obrázků
print(f"Špatně klasifikováno: {len(wrong)} z {len(yg_test)} testovacích obrázků")

fig, axes = plt.subplots(1, min(10, len(wrong)), figsize=(12, 2))
for ax, i in zip(np.atleast_1d(axes), wrong):
    ax.imshow(Xg_test[i].reshape(8, 8), cmap="gray_r")            # řádek 64 pixelů zpět na obrázek 8 × 8
    ax.set_title(f"skut. {yg_test[i]}\npředp. {yg_pred[i]}", fontsize=9)
    ax.axis("off")
plt.show()

## 3.3 Shlukování – Iris bez označení

Dataset Iris obsahuje měření 150 květů kosatců tří druhů. **Druh schováme** a necháme algoritmus K-means najít 3 skupiny.
(Samotný K-means podrobně probereme na cvičení 06.)

`pd.crosstab` spočítá, kolik květů každého druhu skončilo v jednotlivých shlucích.

In [ ]:
iris = datasets.load_iris(as_frame=True)
X_iris, y_iris = iris.data, iris.target                           # druhy zůstávají stranou

kmeans = KMeans(n_clusters=3, n_init=10, random_state=RSTATE)     # 3 shluky, 10 náhodných startů
clusters = kmeans.fit_predict(X_iris)                             # pozor: žádné y!

fx, fy = "petal length (cm)", "petal width (cm)"                  # dva ze čtyř příznaků pro graf
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
axes[0].scatter(X_iris[fx], X_iris[fy], c=clusters, cmap="viridis", s=20)
axes[0].set_title("Shluky nalezené algoritmem K-means")
axes[1].scatter(X_iris[fx], X_iris[fy], c=y_iris, cmap="viridis", s=20)
axes[1].set_title("Skutečné druhy (K-means je nezná)")
for ax in axes:
    ax.set_xlabel(fx)
axes[0].set_ylabel(fy)
plt.show()

druh = pd.Series(iris.target_names[y_iris], name="druh")           # názvy druhů jen pro popis výsledku
pd.crosstab(druh, pd.Series(clusters, name="shluk"))              # druh × shluk

Čísla shluků jsou libovolná – K-means nezná názvy druhů, jen seskupuje podobné květy.
Tabulka ukazuje, že *setosa* je oddělená dokonale, zatímco *versicolor* a *virginica* se částečně překrývají.

**✏️ Úkol 3.1**

* U každého problému rozhodněte, zda jde o regresi, klasifikaci nebo shlukování:
  předpověď zítřejší teploty, filtrování spamu, seskupení zákazníků podle nákupního chování, rozpoznání dopravní značky, odhad zbývající životnosti ložiska.
* Spusťte K-means na Iris s `n_clusters=2` a `n_clusters=5`. Existuje „správný“ počet shluků, když nemáme označení?

# Část 4 – Evaluace

Číslo jako „správnost 0,95“ samo o sobě nic neříká. Vždy se ptáme: **ve srovnání s čím?** a **je to pro daný problém ta správná metrika?**

## 4.1 Baseline – nejjednodušší možný model

**Baseline** (srovnávací model) je triviální model, který příznaky vůbec nepoužívá:

* regrese – vždy předpoví průměr trénovacích cílových hodnot (`DummyRegressor`),
* klasifikace – vždy předpoví nejčastější třídu (`DummyClassifier`).

Užitečný model musí být jasně lepší než baseline.

## 4.2 Metriky pro regresi

| metrika | vzorec | význam |
|---|---|---|
| MAE | $\frac{1}{n}\sum \lvert y_i - \hat{y}_i \rvert$ | průměrná chyba v jednotkách cílové proměnné |
| RMSE | $\sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2}$ | podobně jako MAE, ale velké chyby penalizuje víc |
| $R^2$ | $1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$ | 1 = dokonalý model, 0 = stejně dobrý jako průměr, < 0 = horší než průměr |

In [ ]:
dummy_reg = DummyRegressor(strategy="mean").fit(Xd_train, yd_train)   # vždy předpoví průměr trénovacích cílů

rows = []                                                         # řádky budoucí tabulky výsledků
for name, model in [("baseline (průměr)", dummy_reg), ("lineární regrese", lin_reg), ("rozhodovací strom", tree_reg)]:
    for split, X_, y_ in [("trénovací", Xd_train, yd_train), ("testovací", Xd_test, yd_test)]:
        p = model.predict(X_)                                     # předpovědi pro danou část dat
        rows.append({
            "model": name, "data": split,
            "MAE": metrics.mean_absolute_error(y_, p),
            "RMSE": metrics.root_mean_squared_error(y_, p),
            "R2": metrics.r2_score(y_, p),
        })
pd.DataFrame(rows).set_index(["model", "data"]).round(2)          # tabulka s dvouúrovňovým indexem

## 4.3 Metriky pro klasifikaci

U binárního klasifikátoru padne každá předpověď do jednoho ze čtyř polí **matice záměn** (confusion matrix):

| | předpověď negativní | předpověď pozitivní |
|---|---|---|
| **skutečně negativní** | TN – správně negativní | FP – falešně pozitivní (planý poplach) |
| **skutečně pozitivní** | FN – falešně negativní (přehlédnutý případ) | TP – správně pozitivní |

| metrika | vzorec | na co odpovídá |
|---|---|---|
| správnost (accuracy) | $\frac{TP + TN}{TP + TN + FP + FN}$ | jak často má model pravdu? |
| přesnost (precision) | $\frac{TP}{TP + FP}$ | když model řekne „pozitivní“, jak často má pravdu? |
| úplnost (recall, senzitivita) | $\frac{TP}{TP + FN}$ | kolik skutečně pozitivních případů model našel? |
| F1 | $2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$ | harmonický průměr přesnosti a úplnosti |

In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)   # vždy nejčastější třída
predictions = {"baseline": dummy_clf.predict(X_test), "náhodný les": forest.predict(X_test)}

rows = []
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, p) in zip(axes, predictions.items()):
    metrics.ConfusionMatrixDisplay.from_predictions(              # matice záměn z hotových předpovědí
        y_test, p, labels=[0, 1], display_labels=["nezhoubný", "zhoubný"], ax=ax, colorbar=False
    )
    ax.set(title=name, xlabel="předpovězená třída", ylabel="skutečná třída")
    rows.append({                                                 # metriky pro pozitivní třídu 1 = zhoubný
        "model": name,
        "accuracy": metrics.accuracy_score(y_test, p),
        "precision": metrics.precision_score(y_test, p, zero_division=0),
        "recall": metrics.recall_score(y_test, p),
        "F1": metrics.f1_score(y_test, p, zero_division=0),
    })
plt.tight_layout()
plt.show()
display(pd.DataFrame(rows).set_index("model").round(3))

Baseline má správnost přes 60 % – aniž by se podíval na jediný příznak. A jeho úplnost (recall) je 0: nenajde ani jeden zhoubný nádor.

**Která chyba je tady horší?** Falešně pozitivní předpověď znamená zbytečné vyšetření navíc. Falešně negativní znamená přehlédnutou rakovinu.
Pro tento problém je klíčová metrika **úplnost (recall)** pro třídu „zhoubný“. Volba metriky je rozhodnutí o *problému*, ne o modelu.

## 4.4 Proč správnost může klamat – nevyvážená data

Představte si dataset pro detekci poruch, ve kterém jsou poruchou jen 3 % vzorků.

In [ ]:
X_imb, y_imb = datasets.make_classification(                      # umělá binární úloha
    n_samples=5000, n_features=10, n_informative=4, weights=[0.97, 0.03], random_state=RSTATE
)
Xi_train, Xi_test, yi_train, yi_test = train_test_split(X_imb, y_imb, test_size=0.3, stratify=y_imb, random_state=RSTATE)
print(f"Podíl pozitivní třídy (poruch): {y_imb.mean():.3f}")

rows = []
for name, model in [("baseline", DummyClassifier(strategy="most_frequent")),
                    ("náhodný les", RandomForestClassifier(random_state=RSTATE))]:
    p = model.fit(Xi_train, yi_train).predict(Xi_test)            # učení a předpověď v jednom řádku
    rows.append({"model": name,
                 "accuracy": metrics.accuracy_score(yi_test, p),
                 "recall": metrics.recall_score(yi_test, p),
                 "F1": metrics.f1_score(yi_test, p, zero_division=0)})
display(pd.DataFrame(rows).set_index("model").round(3))

Model, který poruchu **nikdy** nenajde, má správnost asi 97 %. Náhodný les je ve správnosti jen o málo lepší, ale aspoň část poruch skutečně najde.
U nevyvážených dat se proto vždy dívejte na matici záměn, přesnost a úplnost (nebo F1), ne jen na správnost.

**✏️ Úkol 4.1** – Pro screening rakoviny je důležitější úplnost (recall). Pro jaký problém by byla důležitější přesnost (precision)? (Nápověda: spamový filtr a důležitý e-mail ve spamu.)

## 4.5 Evaluace shlukování

Bez označení nemáme „správnou odpověď“, se kterou bychom mohli výsledek porovnat. Můžeme jen měřit, jak kompaktní a dobře oddělené shluky jsou:

* **inercie** (inertia) – součet čtverců vzdáleností bodů od středu jejich shluku (nižší = kompaktnější, ale s počtem shluků vždy klesá),
* **silhouette skóre** – od −1 do 1, vyšší = body jsou blízko vlastnímu shluku a daleko od ostatních.

Pokud skutečné označení náhodou známe (jako u Iris), můžeme s ním výsledek porovnat, např. pomocí **Adjusted Rand Index** (1 = shodné rozdělení, ≈ 0 = náhodné).

In [ ]:
rows = []
for k in range(2, 11):                                            # různé počty shluků
    km = KMeans(n_clusters=k, n_init=10, random_state=RSTATE).fit(X_iris)   # učení bez druhů
    rows.append({"k": k,
                 "inercie": km.inertia_,
                 "silhouette": metrics.silhouette_score(X_iris, km.labels_),
                 "ARI": metrics.adjusted_rand_score(y_iris, km.labels_)})   # jediná metrika se skutečnými druhy
cluster_results = pd.DataFrame(rows).set_index("k")

cluster_results.plot(subplots=True, layout=(1, 3), figsize=(15, 4), marker="o", legend=False,
                     title=["Inercie", "Silhouette skóre", "Adjusted Rand Index (používá skutečné druhy)"])
plt.tight_layout()
plt.show()

Zajímavé: silhouette skóre je nejvyšší pro **2** shluky, protože se *versicolor* a *virginica* překrývají – přitom skutečné druhy jsou 3.
Metriky pro učení bez učitele měří geometrii dat, ne „pravdu“.

# Shrnutí – kontrolní seznam pro každý ML experiment

1. **Data** – Co je vzorek? Jaké jsou příznaky a cílová proměnná? Mají data označení? Jsou vyvážená?
2. **Úloha** – Regrese, klasifikace, nebo shlukování?
3. **Rozdělení** – Testovací data odložte stranou dřív, než začnete cokoli ladit. Nastavení porovnávejte na validačních datech.
4. **Baseline** – Jak dobrý je triviální model?
5. **Metrika** – Které chyby jsou pro *tento* problém důležité? Podle toho zvolte metriku.
6. **Generalizace** – Porovnejte výkon na trénovacích a testovacích datech: podučení, nebo přeučení?

**Co přijde později v semestru:** předzpracování dat (cvičení 08); systematický výběr modelu podle validační metriky, křížová validace a volba rozhodovacího prahu (cvičení 07).

Pokračujte [samostatnou prací](Independent_Work_CZ.ipynb).